# 01 · Beliefs are probability vectors—not shared omniscience

This notebook derives and executes the core model. A character sees only its own observations and received messages. The designer has a separate privileged inspector.

**What to look for:** twenty copies of one report should not have the same effect as two independent measurements. All calculations below use the same package as the game.

The complete assumptions and derivation are in [MATH.md](../docs/MATH.md).

In [ ]:
from pathlib import Path
import sys, json, os
ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "belieftown").exists():
    raise RuntimeError("Run this notebook from the source repository or its notebooks directory.")
sys.path.insert(0, str(ROOT / "src"))
RUN_ROOT = Path(os.environ["BELIEFTOWN_RUN_ROOT"]).resolve() if os.environ.get("BELIEFTOWN_RUN_ROOT") else None
DATA = RUN_ROOT / "data" if RUN_ROOT else ROOT / "benchmarks" / "data"
CHECKPOINTS = RUN_ROOT / "checkpoints" if RUN_ROOT else ROOT / "artifacts" / "checkpoints"
REPORT = RUN_ROOT / "evaluation" if RUN_ROOT else ROOT / "artifacts" / "evaluation"
AUDIT_DIR = RUN_ROOT if RUN_ROOT else ROOT / "artifacts"
OUTPUT = AUDIT_DIR / "notebook_run"
OUTPUT.mkdir(parents=True, exist_ok=True)
import numpy as np
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
from IPython.display import display, HTML

## A finite belief state

For $L$ rooms, the character maintains $\mathbf b=(P(X=1),\ldots,P(X=L))^\top$, with nonnegative entries summing to one. A noisy report has likelihood $q$ in the reported room and $(1-q)/(L-1)$ elsewhere.

Bayes’ rule is $b'(x)\propto P(y\mid x)b(x)$. Root IDs identify shared measurements; they do **not** assert that arbitrary real-world reports are independent.

In [ ]:
from dataclasses import replace
from belieftown.schema import Observation
from belieftown.beliefs import filter_belief, brute_force_belief, entropy

root = Observation("original", "mira", 0, 1, 0.8)
prior = np.ones(4) / 4
one = filter_belief(prior, [], [root], 0)
copies = filter_belief(prior, [], [root] * 20, 0)
independent = filter_belief(prior, [], [root, replace(root, id="independent")], 0)
np.testing.assert_array_equal(one, copies)
assert np.isclose(independent[1], 0.64 / (0.64 + 3 * (0.2 / 3) ** 2))
print("One report:", one.round(6))
print("Twenty copies:", copies.round(6))
print("Two independent reports:", independent.round(6))

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(["One report", "Twenty copies", "Two independent"], [one[1], copies[1], independent[1]])
plt.ylim(0, 1.05)
plt.ylabel("Probability assigned to the reported room")
plt.title("Independent corroboration is different from repetition")
plt.tight_layout()
plt.show()

## A late message is evidence about the past

Let $T_t[x',x]=P(X_{t+1}=x'\mid X_t=x)$. After receiving old evidence, insert it at its observation time and recompute forward. Do not pretend it describes the present.

The implementation uses log probabilities internally, including a log-sum-exp prediction step, so very confident but conflicting evidence does not prematurely destroy support through underflow.

In [ ]:
# The parcel deterministically swaps between two rooms after turn zero.
swap = (((0.0, 1.0), (1.0, 0.0)),)
old = Observation("old", "mira", 0, 0, 0.9)
late_belief = filter_belief([0.5, 0.5], swap, [old], 1)
reference = brute_force_belief([0.5, 0.5], swap, [old], 1)
np.testing.assert_allclose(late_belief, reference)
print("Current belief after the old report:", late_belief)
assert np.isclose(late_belief[1], 0.9)

## The actual game scene

At turn four the Courier has an old workshop report and a direct observation that the Hall is empty. Mira has an undelivered library observation. No policy reads the hidden parcel location.

In [ ]:
from belieftown.scenarios import demo_scene
from belieftown.world import Episode
scene = demo_scene()
episode = Episode(scene)
snapshot = episode.at(scene.search_start)
for agent in scene.actors:
    print(agent.name, snapshot.belief(agent.id).round(4), "next:", scene.rooms[snapshot.plan(agent.id).destination])
assert "new-library" not in snapshot.known("courier")
assert not hasattr(snapshot.view("courier"), "truth")

## Expected-cost route planning is a matrix–vector product

For each room ordering $\sigma$, let $c_\sigma(x)$ be the travel-plus-inspection cost incurred before finding the parcel if it is in room $x$. Then

$$J(\sigma\mid\mathbf b)=\mathbf c_\sigma^\top\mathbf b,\qquad \mathbf j=\mathbf C\mathbf b.$$

With four rooms, there are $4!=24$ orders. This is an exact plan among fixed orders for the stationary search phase, not a globally optimal multi-agent POMDP policy.

In [ ]:
from belieftown.planner import route_matrix
orders, C = route_matrix(len(scene.rooms), scene.edges, snapshot.positions["courier"], scene.inspection_cost)
b = snapshot.belief("courier")
expected_costs = C @ b
best = int(np.argmin(expected_costs))
print("Cost-matrix shape:", C.shape)
print("Chosen route:", " → ".join(scene.rooms[i] for i in orders[best]))
print("Expected cost:", expected_costs[best])
print("Belief entropy, in nats:", entropy(b))
assert orders[best] == snapshot.plan("courier").order

**Result:** the incorrect search destination follows from available evidence and the route objective. It is not an arbitrary dialogue prompt. Continue to notebook 02 to change that evidence without changing the world.